In [7]:
%pip install transformers datasets peft accelerate bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You should consider upgrading via the '/Users/danilkladnitsky/.pyenv/versions/3.10.4/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [9]:
import json

# Paths
ORIGINAL_DATASET_PATH = "datasets/hsk1-dataset.json"
SAVED_DATASET_PATH = "datasets/hsk1-dataset-formatted.json"

# Load original dataset
with open(ORIGINAL_DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert format
formatted_data = []
for entry in data:
    labeled = entry["labeled_sentence"]
    try:
        # Extract the word between “ and ”
        word = labeled.split("“")[1].split("”")[0]
        prompt = f"请用词语“{word}”造句："
        completion = entry["original_sentence"]
        formatted_data.append({
            "prompt": prompt,
            "completion": completion,
            "word": word
        })
    except IndexError:
        print(f"Skipping entry due to format error: {entry}")

# Save new dataset
with open(SAVED_DATASET_PATH, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"Formatted {len(formatted_data)} entries and saved to {SAVED_DATASET_PATH}")

Formatted 8355 entries and saved to datasets/hsk1-dataset-formatted.json


In [11]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

# 1. Load model and tokenizer
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # For padding
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True)
model.to("cpu")

# 2. Apply LoRA
lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

# 3. Tiny dataset for testing
DATASET_PATH = "datasets/hsk1-dataset.json"
examples = json.load(open(DATASET_PATH))

def format(example):
    return {
        "text": f"### Instruction:\n{example['prompt']}\n### Input:\n{example['word']}\n### Response:\n{example['completion']}"
    }

dataset = Dataset.from_list(examples).map(format)

# 4. Tokenization
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize)

# 5. Training setup
training_args = TrainingArguments(
    output_dir="./qwen-cpu-test",
    per_device_train_batch_size=2,  # increase if you have enough RAM
    gradient_accumulation_steps=8,  # simulate effective batch size of 16
    num_train_epochs=3,             # 3–5 is a good starting point
    learning_rate=2e-4,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    no_cuda=True
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

# 6. Train!
if __name__ == "__main__":
    threads_num = torch.get_num_threads()
    torch.set_num_threads(threads_num)  # Use appropriate number for your CPU
    trainer.train()
    model.save_pretrained("models/qwen2.5-lora-test")
    tokenizer.save_pretrained("models/wen2.5-lora-test")

Map: 100%|██████████| 8355/8355 [00:00<00:00, 9744.45 examples/s]
/Users/danilkladnitsky/.pyenv/versions/3.10.4/lib/python3.10/site-packages/transformers/training_args.py:1595: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
/var/folders/p9/gl1s91rn2fv_77662_261gb80000gn/T/ipykernel_71906/1292929865.py:58: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss


KeyboardInterrupt: 

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Paths
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "./qwen2.5-lora-test"  # Your LoRA adapter

# Load base model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(base_model_id, trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.to("cpu")
model.eval()

# Inference function
def generate_response(instruction, user_input, max_new_tokens=50):
    prompt = f"### Instruction:\n{instruction}\n### Input:\n{user_input}\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cpu")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
    
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\n📝 Generated Output:\n")
    print(result.split("### Response:\n")[-1].strip())

# 🔍 Example usage
if __name__ == "__main__":
    generate_response("Generate three sentences with this word using hsk 1 vocabulary: ", "现在")


📝 Generated Output:

现在 is a verb, it means "the present" in English. It can be used to talk about things that are still existing or happening now. Here are some sentences using HSK 1 vocabulary:

1. 好了。
2.


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

# === Paths ===
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct-GPTQ-Int4"
adapter_path = "qwen2.5-lora-test"
output_path = "qwen2.5-merged-model"

# === Load base model and LoRA adapter ===
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, trust_remote_code=True)
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)

# === Merge LoRA weights into base ===
print("Merging weights...")
model = model.merge_and_unload()

# === Save merged model ===
print(f"Saving merged model to {output_path} ...")
model.save_pretrained(output_path)

# Also save tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
tokenizer.save_pretrained(output_path)

print("✅ Merge complete. You can now load the model with AutoModelForCausalLM directly.")

Loading base model...


ImportError: Loading a GPTQ quantized model requires optimum (`pip install optimum`)

In [9]:
pip install -U bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You should consider upgrading via the '/Users/danilkladnitsky/.pyenv/versions/3.10.4/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
